# Extract Text from Product Documentation PDFs (v2)
Optimized for Databricks UC workspaces using Spark binary file reader.

**Configuration:**
- Source: `/Volumes/llmagent/dev/data_volume/01_Data_Files/product_docs/`
- Target: `llmagent.dev.product_details`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col, udf, regexp_replace
from io import BytesIO
import PyPDF2
import os

spark = SparkSession.builder.appName("PDFExtraction").getOrCreate()

# Configuration
VOLUME_PATH = "/Volumes/llmagent/dev/data_volume/01_Data_Files/product_docs"
CATALOG = "llmagent"
SCHEMA = "dev"
TABLE_NAME = "product_details"
FULL_TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

print(f"Target Table: {FULL_TABLE_NAME}")
print(f"Volume Path: {VOLUME_PATH}")

In [ ]:
# Define PDF text extraction function
def extract_pdf_text(pdf_bytes):
    if pdf_bytes is None:
        return ""
    try:
        pdf_file = BytesIO(pdf_bytes)
        pdf_reader = PyPDF2.PdfReader(pdf_file)
        text_content = []
        for page in pdf_reader.pages:
            text = page.extract_text()
            if text:
                text_content.append(text)
        return "\n".join(text_content)
    except:
        return ""

# Register UDF
extract_udf = udf(extract_pdf_text, StringType())

print("✓ PDF extraction UDF registered")

In [ ]:
# Read PDF files as binary format
df_raw = spark.read.format("binaryFile") \
    .option("pathGlobFilter", "*.pdf") \
    .load(VOLUME_PATH)

print(f"✓ Found {df_raw.count()} PDF files")
df_raw.display() if hasattr(spark, 'display') else df_raw.show(5)

In [ ]:
# Extract filename without extension and extract text
df_extracted = df_raw.select(
    regexp_replace(col("path"), r'^.*/([^/]+)\.pdf$', '$1').alias("product_name"),
    extract_udf(col("content")).alias("product_doc")
)

# Filter out PDFs with no extracted text
df_extracted = df_extracted.filter(col("product_doc") != "")

print(f"✓ Extracted text from {df_extracted.count()} PDFs")

In [ ]:
# Create table if not exists
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {FULL_TABLE_NAME} (
        product_name STRING,
        product_doc STRING
    )
    USING DELTA
""")

print(f"✓ Table {FULL_TABLE_NAME} ready")

In [ ]:
# Write to table
df_extracted.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(FULL_TABLE_NAME)

row_count = spark.sql(f"SELECT COUNT(*) as count FROM {FULL_TABLE_NAME}").collect()[0]['count']
print(f"✓ Successfully wrote {row_count} rows to {FULL_TABLE_NAME}")

In [ ]:
# Show sample results
display(spark.sql(f"""
    SELECT
        product_name,
        LENGTH(product_doc) as text_length,
        SUBSTR(product_doc, 1, 200) as preview
    FROM {FULL_TABLE_NAME}
    LIMIT 5
"""))

In [ ]:
# Show statistics
display(spark.sql(f"""
    SELECT
        COUNT(*) as total_records,
        MIN(LENGTH(product_doc)) as min_text_length,
        MAX(LENGTH(product_doc)) as max_text_length,
        ROUND(AVG(LENGTH(product_doc)), 0) as avg_text_length
    FROM {FULL_TABLE_NAME}
"""))